In [ ]:
from pathlib import Path

MODEL='ETLTC F'

DATASET_PATH = Path('preprocessed_data/kkanji2')

CLASSES_SHOWN = 5

RESULT_PATH = Path('class_accuracy/' + MODEL)

RESULT_PATH.mkdir(parents=True, exist_ok=True)

GET_CLASS_DISTRIBUTION = True

In [ ]:
import pickle
import csv
import json
import tqdm

if (GET_CLASS_DISTRIBUTION):
    class_frequency_map = {} #[class] = [samples_in_class]
    with open(str(DATASET_PATH / 'train_labels.json'), 'r') as fp:
        train_dict = json.load(fp)
    for sample in tqdm.tqdm(train_dict.values(), total=len(train_dict.values()), desc=f'Calculating class distribution'):
        word = sample.strip()
        if (word not in class_frequency_map):
            class_frequency_map[word] = 0
        class_frequency_map[word] += 1
    with open(f'class_accuracy/class_frequency_map.pkl', 'wb') as f:
            pickle.dump(class_frequency_map, f)
    with open(f'class_accuracy/class_frequency_table.csv', 'w') as f:
        writer = csv.writer(f)
        writer.writerow(class_frequency_map.keys())
        writer.writerow(class_frequency_map.values())
    
with open(f'class_accuracy/class_frequency_map.pkl', 'rb') as f:
    class_frequency_map = pickle.load(f)

In [ ]:
print("Getting class distribution data")
SAMPLE_THRESHOLD = 10
to_exclude_list = []
few_char_count = 0
for char, count in class_frequency_map.items():
    if char == '大': print(count)
    if count < SAMPLE_THRESHOLD: 
        few_char_count += 1
        to_exclude_list.append(char)

print(f"{few_char_count} classes have less than {SAMPLE_THRESHOLD} samples out of total {len(class_frequency_map.keys())}")

items = class_frequency_map.items()
least_popular = sorted(items, key=lambda x: x[1], reverse=False)
print(f"{SAMPLE_THRESHOLD} least popular classes")
for item in least_popular[:10]:
    print(f"Character: {item[0]}, Character count: {item[1]}")

In [ ]:
import json
from pathlib import Path
from util.data_processing import get_words_list


with open(str(DATASET_PATH / 'test_labels.json'), 'r') as fp:
    test_dict = json.load(fp)

    
test_words_files = list(test_dict.items())
test_words = get_words_list(test_words_files)


In [ ]:
from util.model import get_model
from dtrocr.config import DTrOCRConfig

config = DTrOCRConfig(
    # attn_implementation='flash_attention_2'
)
model = get_model(config, MODEL)

In [ ]:
from util.model import check_class_accuracy

_, class_map = check_class_accuracy(MODEL, model, test_words, to_exclude_list)

In [ ]:
def get_stats(class_map: dict):
    most_popular = []
    best_acc = []
    worst_acc = []
    items = class_map.items()
    most_popular = sorted(items, key=lambda x: x[1][0], reverse=True)
    best_acc = sorted(items, key=lambda x: x[1][1], reverse=True)
    worst_acc = sorted(items, key=lambda x: x[1][1], reverse=False)
    
    print(f"{CLASSES_SHOWN} most popular classes")
    for item in most_popular[:CLASSES_SHOWN+20]:
        print(f"Character: {item[0]}, Accuracy: {item[1][1]}, Character count: {item[1][0]}")
    
    # print(f"{CLASSES_SHOWN} classes with highest accuracy")
    # for item in best_acc[:CLASSES_SHOWN]:
    #     print(f"Character: {item[0]}, Accuracy: {item[1][1]}, Character count: {item[1][0]}")
    
    # print(f"{CLASSES_SHOWN} classes with lowest accuracy")
    # for item in worst_acc[:CLASSES_SHOWN]:
    #     print(f"Character: {item[0]}, Accuracy: {item[1][1]}, Character count: {item[1][0]}")
        

get_stats(class_map)

In [ ]:
import pickle
import csv

with open(f'{RESULT_PATH}/test_class_map.pkl', 'wb') as f:
    pickle.dump(class_map, f)

with open(f'{RESULT_PATH}/test_class_table.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerow(class_map.keys())
    writer.writerow(class_map.values())
    
# with open(f'class_accuracy/{MODEL}/test_class_map.pkl', 'rb') as f:
#     class_map = pickle.load(f)


All models

In [ ]:
def get_stats(class_map: dict):
    most_popular = []
    best_acc = []
    worst_acc = []
    items = class_map.items()
    most_popular = sorted(items, key=lambda x: x[1][0], reverse=True)
    best_acc = sorted(items, key=lambda x: x[1][1], reverse=True)
    worst_acc = sorted(items, key=lambda x: x[1][1], reverse=False)
    
    print(f"{CLASSES_SHOWN} most popular classes")
    for item in most_popular[:CLASSES_SHOWN]:
        print(f"Character: {item[0]}, Accuracy: {item[1][1]}, Character count: {item[1][0]}")
        
    print(f"{CLASSES_SHOWN} classes with highest accuracy")
    for item in best_acc[:CLASSES_SHOWN]:
        print(f"Character: {item[0]}, Accuracy: {item[1][1]}, Character count: {item[1][0]}")
        
    print(f"{CLASSES_SHOWN} classes with lowest accuracy")
    for item in worst_acc[:CLASSES_SHOWN]:
        print(f"Character: {item[0]}, Accuracy: {item[1][1]}, Character count: {item[1][0]}")

In [ ]:
import json
from pathlib import Path
from util.data_processing import get_words_list
from util.model import get_model
from dtrocr.config import DTrOCRConfig
from util.model import check_class_accuracy
import pickle
import csv
import tqdm

models_list = ['ETLTC A', 'ETLTC B', 'ETLTC C', 'ETLTC D', 'ETLTC E', 'ETLTC F']

for model_name in models_list:
    print(f'Evaluating class accuracy statistics for model {model_name}')
    with open(str(DATASET_PATH / 'test_labels.json'), 'r') as fp:
        test_dict = json.load(fp)

        
    test_words_files = list(test_dict.items())
    test_words = get_words_list(test_words_files)



    config = DTrOCRConfig(
        # attn_implementation='flash_attention_2'
    )
    model = get_model(config, model_name)



    _, class_map = check_class_accuracy(model_name, model, test_words, to_exclude_list)

    get_stats(class_map)



    with open(f'{RESULT_PATH}/test_class_map.pkl', 'wb') as f:
        pickle.dump(class_map, f)

    with open(f'{RESULT_PATH}/test_class_table.csv', 'w') as f:
        writer = csv.writer(f)
        writer.writerow(class_map.keys())
        writer.writerow(class_map.values())
        
    # with open(f'class_accuracy/{MODEL}/test_class_map.pkl', 'rb') as f:
    #     class_map = pickle.load(f)

<H3>Evaluating sample count's impact on perfromance.</H3>

In [ ]:
import pickle
import csv
import json
import tqdm

if (GET_CLASS_DISTRIBUTION):
    class_frequency_map = {} #[class] = [samples_in_class]
    with open(str(DATASET_PATH / 'train_labels.json'), 'r') as fp:
        train_dict = json.load(fp)
    for sample in tqdm.tqdm(train_dict.values(), total=len(train_dict.values()), desc=f'Calculating class distribution'):
        word = sample.strip()
        if (word not in class_frequency_map):
            class_frequency_map[word] = 0
        class_frequency_map[word] += 1

In [ ]:
import json
from pathlib import Path
from util.data_processing import get_words_list
from util.model import get_model
from dtrocr.config import DTrOCRConfig
from util.model import check_class_accuracy
import pickle
import csv
import tqdm

threshold_list = [5, 10, 20, 50]

for threshold in threshold_list:
    print("Getting class distribution data")
    to_exclude_list = []
    few_char_count = 0
    for char, count in class_frequency_map.items():
        if count < threshold: 
            few_char_count += 1
            to_exclude_list.append(char)

    print(f"{few_char_count} classes have less than {threshold} samples out of total {len(class_frequency_map.keys())}")

    print(f'Evaluating class accuracy statistics for model {model_name}')
    with open(str(DATASET_PATH / 'test_labels.json'), 'r') as fp:
        test_dict = json.load(fp)

        
    test_words_files = list(test_dict.items())
    test_words = get_words_list(test_words_files)



    config = DTrOCRConfig(
        # attn_implementation='flash_attention_2'
    )
    model = get_model(config, model_name)



    _, class_map = check_class_accuracy(model_name, model, test_words, to_exclude_list)

    get_stats(class_map)



    with open(f'{RESULT_PATH}/test_class_map_{threshold}.pkl', 'wb') as f:
        pickle.dump(class_map, f)

    with open(f'{RESULT_PATH}/test_class_table_{threshold}.csv', 'w') as f:
        writer = csv.writer(f)
        writer.writerow(class_map.keys())
        writer.writerow(class_map.values())
        
    # with open(f'class_accuracy/{MODEL}/test_class_map.pkl', 'rb') as f:
    #     class_map = pickle.load(f)